# Day 2 Answer Key: Self-Attention, Multi-Head Attention, and the KV Cache

**COMP 395 — Deep Learning | Transformers Unit**

⚠️ **ANSWER KEY — Do not distribute to students.**

This notebook contains all TODO solutions and model answers for reflections. Uses GloVe-50d embeddings.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import os
import urllib.request
import zipfile

# Reproducibility
torch.manual_seed(395)
np.random.seed(395)

# Plot style
plt.rcParams.update({
    'figure.figsize': (8, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
})

# Device selection (same pattern as Lab 6)
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

# ── Load GloVe-50d (same vectors you used in the Word Embeddings activity) ──
glove_path = 'glove.6B.50d.txt'

if not os.path.exists(glove_path):
    print('Downloading GloVe-50d (this may take a minute)...')
    url = 'https://nlp.stanford.edu/data/glove.6B.zip'
    urllib.request.urlretrieve(url, 'glove.6B.zip')
    with zipfile.ZipFile('glove.6B.zip', 'r') as z:
        z.extract('glove.6B.50d.txt')
    print('Done!')

def load_glove(path):
    """Load GloVe vectors into a dictionary: word -> tensor."""
    embeddings = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vec = torch.tensor([float(x) for x in parts[1:]])
            embeddings[word] = vec
    return embeddings

glove = load_glove(glove_path)
print(f'Loaded {len(glove):,} GloVe vectors, dimension = {len(next(iter(glove.values())))}')

def get_embeddings(words, glove_dict):
    """Look up GloVe embeddings for a list of words (lowercase)."""
    vecs = []
    for w in words:
        key = w.lower()
        if key in glove_dict:
            vecs.append(glove_dict[key])
        else:
            print(f"  Warning: '{w}' not in GloVe, using random vector")
            vecs.append(torch.randn(50))
    return torch.stack(vecs)

---

## Reuse from Day 1

Here's the `scaled_dot_product_attention` function you built yesterday, plus our GloVe-50d embeddings — the same real word vectors you used in the Word Embeddings activity.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    """
    Scaled dot-product attention (from Day 1).
    
    Args:
        Q: Query matrix, shape (n_q, d_k) or (batch, n_q, d_k)
        K: Key matrix,   shape (n_k, d_k) or (batch, n_k, d_k)
        V: Value matrix,  shape (n_k, d_v) or (batch, n_k, d_v)
    
    Returns:
        output:  shape (n_q, d_v)
        weights: shape (n_q, n_k)
    """
    d_k = Q.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights

In [ ]:
def plot_attention_weights(weights, title='Attention Weights', 
                           row_labels=None, col_labels=None):
    """Heatmap of attention weights (reused from Day 1)."""
    if weights.dim() > 2:
        weights = weights.squeeze(0)
    w = weights.detach().numpy()
    
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(w, cmap='YlOrRd', aspect='equal', vmin=0, vmax=w.max())
    
    if row_labels:
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels)
    if col_labels:
        ax.set_xticks(range(len(col_labels)))
        ax.set_xticklabels(col_labels, rotation=45, ha='right')
    
    # Annotate cells
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            color = 'white' if w[i, j] > w.max() * 0.6 else 'black'
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', 
                    fontsize=9, color=color)
    
    ax.set_xlabel('Key (attending to)')
    ax.set_ylabel('Query (attending from)')
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()
    plt.show()

---

## Part 1: Permutation Equivariance Discovery (10 min)

Let's start with an experiment. We'll run self-attention on a sentence and on a **scrambled** version of the same sentence, and see what happens.

This time we're using **real GloVe-50d embeddings** — the same vectors from the Word Embeddings activity. No hand-crafted numbers.

In "self-attention," we use the same input for Q, K, and V. For this discovery, we'll use **simplified** self-attention (no learned projections — just $Q = K = V = X$), which is what you did in Part 4 of Day 1.

### The Keys/Are Sentence

We'll look up real GloVe-50d embeddings for these words — the same vectors from the Word Embeddings activity, not hand-crafted ones.

In [ ]:
# The keys/are sentence with real GloVe-50d embeddings
words = ['the', 'keys', 'to', 'the', 'cabinet', 'are']

embeddings = get_embeddings(words, glove)

print(f'Embeddings shape: {embeddings.shape}  (n_tokens={len(words)}, d={embeddings.shape[1]})')
print(f'\nThese are real GloVe vectors — "keys" and "cabinet" are genuinely')
print(f'close in embedding space, not because we rigged it.')
print(f'\nCosine similarity (keys, cabinet): {F.cosine_similarity(embeddings[1], embeddings[4], dim=0):.3f}')
print(f'Cosine similarity (keys, to):      {F.cosine_similarity(embeddings[1], embeddings[2], dim=0):.3f}')

### Step 1: Self-attention on the original sentence

Run simplified self-attention (Q = K = V = X) and visualize the weights.

In [ ]:
# Self-attention: Q = K = V = embeddings
output_orig, weights_orig = scaled_dot_product_attention(embeddings, embeddings, embeddings)

plot_attention_weights(weights_orig, title='Self-Attention: Original Order', 
                       row_labels=words, col_labels=words)

### Step 2: Scramble the word order

Now let's randomly shuffle the words and their embeddings.

In [ ]:
# Create a random permutation
torch.manual_seed(42)
perm = torch.randperm(len(words))
print(f'Permutation: {perm.tolist()}')

# Apply the permutation to both words and embeddings
scrambled_words = [words[i] for i in perm]
scrambled_embeddings = embeddings[perm]

print(f'Original:  {words}')
print(f'Scrambled: {scrambled_words}')

### ✏️ Prediction (fill in BEFORE running the next cell)

**Before running self-attention on the scrambled sentence:**

1. Will the attention weights be the same, completely different, or related in some systematic way?

   *The weights between each pair of words will be the same — just rearranged in the matrix. Scrambling the order changes which row/column each word occupies, but the dot product between "are" and "keys" only depends on their embeddings, not their positions.*

2. Specifically: in the original, "are" attended to "keys" with some weight $\alpha$. After scrambling, will the attention weight between "are" and "keys" change?

   *No. The weight depends only on the dot product of their embedding vectors, which doesn't change when we move them to different positions.*

### Step 3: Self-attention on the scrambled sentence

In [ ]:
# Self-attention on scrambled embeddings
output_scram, weights_scram = scaled_dot_product_attention(
    scrambled_embeddings, scrambled_embeddings, scrambled_embeddings
)

plot_attention_weights(weights_scram, title='Self-Attention: Scrambled Order',
                       row_labels=scrambled_words, col_labels=scrambled_words)

### Step 4: Compare directly

Let's check whether the attention weight between each pair of words changed.

In [ ]:
# Build a comparison: for each pair of words, show original vs scrambled weight
print("Comparing attention weights between word pairs:")
print(f"{'Query':<10s} {'Key':<10s} {'Original':>10s} {'Scrambled':>10s} {'Diff':>10s}")
print("-" * 52)

for i, w_i in enumerate(words):
    for j, w_j in enumerate(words):
        orig_weight = weights_orig[i, j].item()
        # Find where w_i and w_j ended up in the scrambled order
        scram_i = (perm == i).nonzero().item()
        scram_j = (perm == j).nonzero().item()
        scram_weight = weights_scram[scram_i, scram_j].item()
        diff = abs(orig_weight - scram_weight)
        if diff > 0.001:  # only show if there's a difference
            print(f'{w_i:<10s} {w_j:<10s} {orig_weight:>10.4f} {scram_weight:>10.4f} {diff:>10.4f}')

# Check if any weights actually changed
max_diff = 0
for i in range(len(words)):
    for j in range(len(words)):
        scram_i = (perm == i).nonzero().item()
        scram_j = (perm == j).nonzero().item()
        diff = abs(weights_orig[i, j].item() - weights_scram[scram_i, scram_j].item())
        max_diff = max(max_diff, diff)

print(f"\nMaximum difference in any attention weight: {max_diff:.6f}")
if max_diff < 1e-5:
    print("The weights are IDENTICAL (just reordered). Self-attention is permutation equivariant!")
else:
    print("Something unexpected happened — the weights differ.")

### ✏️ Was your prediction correct?

1. Were the attention weights the same after scrambling? Why or why not?

   *Yes — the maximum difference was essentially zero. The weights are identical because self-attention computes dot products between pairs of embedding vectors, and dot products don't depend on the order of the inputs. Reordering the sequence just permutes the rows and columns of the weight matrix.*

2. What does this tell us about self-attention's understanding of word order?

   *Self-attention has no understanding of word order whatsoever. It treats the input as a set, not a sequence. The technical term is "permutation equivariance" — permute the inputs, and the outputs permute in the same way.*

3. "Dog bites man" vs "Man bites dog" — would self-attention distinguish these? Why is this a problem?

   *No, self-attention would produce the same attention weights for both sentences (just with rows/columns reordered). This is a serious problem because these sentences have opposite meanings. Subject-verb-object order is critical for understanding language, and self-attention by itself cannot capture it.*

---

## Part 2: Sinusoidal Positional Encoding (10 min)

The fix: **add position information** to the embeddings before computing attention.

$$\tilde{\mathbf{x}}_i = \mathbf{x}_i + \mathbf{p}_i$$

The original transformer uses sinusoidal positional encoding:

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

where $pos$ is the position index and $i$ is the dimension index.

Note: the PE is added **before** the Q, K, V projections. That means position information flows into the keys and values — and later, into the KV cache. The cache doesn't just store "what" each token means; it stores "what at which position."

### Task 2.1: Implement sinusoidal positional encoding

Fill in the function below. The key steps are:
1. Create a matrix of position indices × dimension indices
2. Compute the frequency term $10000^{2i/d}$
3. Apply $\sin$ to even columns and $\cos$ to odd columns

In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """
    Compute sinusoidal positional encoding.
    
    Args:
        max_len: maximum sequence length
        d_model: embedding dimension
    
    Returns:
        PE: tensor of shape (max_len, d_model)
    """
    PE = torch.zeros(max_len, d_model)
    
    # Position indices: shape (max_len, 1)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    
    # Dimension indices for the frequency term: shape (d_model/2,)
    # div_term = 10000^(2i / d_model) = exp(2i * -log(10000) / d_model)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    
    # Apply sin to even indices (0, 2, 4, ...) and cos to odd indices (1, 3, 5, ...)
    PE[:, 0::2] = torch.sin(position * div_term)  # Even columns: sin
    PE[:, 1::2] = torch.cos(position * div_term)  # Odd columns: cos
    
    return PE

### Task 2.2: Visualize the positional encoding as a heatmap

A good positional encoding should show:
- **Fast oscillation** in low dimensions (left side)
- **Slow oscillation** in high dimensions (right side)
- **Unique pattern** for each position (each row looks different)

In [ ]:
# Generate positional encoding for 50 positions, 64 dimensions
PE = sinusoidal_positional_encoding(max_len=50, d_model=64)

print(f'PE shape: {PE.shape}')
print(f'PE[0, :8] (position 0, first 8 dims): {PE[0, :8].tolist()}')

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(PE.numpy(), cmap='RdBu', aspect='auto', interpolation='nearest')

ax.set_xlabel('Dimension')
ax.set_ylabel('Position')
ax.set_title('Sinusoidal Positional Encoding')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

### ✏️ Checkpoint Questions

1. Why do the left columns of the heatmap oscillate faster than the right columns?

   *The frequency decreases as the dimension index $i$ increases. Low dimensions (left) use high-frequency sin/cos waves (short wavelength), so they cycle rapidly across positions. High dimensions (right) use low-frequency waves (long wavelength), so they change slowly. This comes from the $10000^{2i/d}$ denominator — as $i$ grows, the denominator grows, which stretches the wavelength.*

2. Can two different positions have the same encoding? Why or why not?

   *No, not within any practical sequence length. Each dimension oscillates at a different wavelength, and the wavelengths are not rational multiples of each other. For two positions to have identical PE vectors, every sin and cos at every frequency would need to match simultaneously, which requires the positions to be identical.*

3. The slides mentioned a "clock" analogy. How does the heatmap relate to that analogy?

   *The leftmost columns are like a seconds hand — they change value rapidly from one position to the next. The rightmost columns are like an hours hand — they change very slowly. Just as reading all three hands of a clock uniquely identifies a time, reading across all dimensions of the PE uniquely identifies a position.*

### Task 2.3: Re-run the permutation experiment WITH positional encoding

Now let's add positional encoding to our embeddings and see if the attention weights change when we scramble the sentence.

### ✏️ Prediction

**Before running the code below:**

When we add positional encoding to the embeddings, will scrambling the sentence produce different attention weights now?

*Yes. With PE, each word's embedding is modified based on its position in the sequence. The same word ("keys") at position 1 gets a different PE vector added than at position 4. So when we scramble and the words land at different positions, their modified embeddings change, which changes the dot products, which changes the attention weights.*

In [ ]:
# Generate PE for our sentence (6 positions, 50 dimensions to match GloVe)
PE_small = sinusoidal_positional_encoding(max_len=6, d_model=50)

# Add PE to original embeddings
embeddings_with_pe = embeddings + PE_small

# Add PE to scrambled embeddings
# Key insight: the SAME PE is added based on POSITION, not word identity
scrambled_embeddings_with_pe = scrambled_embeddings + PE_small

print("Original embeddings + PE:")
print(f"  'keys' (position 1): embedding + PE[1]")
print(f"Scrambled embeddings + PE:")
print(f"  'keys' is now at position {(perm == 1).nonzero().item()}: embedding + PE[{(perm == 1).nonzero().item()}]")
print(f"\nSame word, different position -> different PE added -> different input to attention!")

In [ ]:
# Self-attention WITH positional encoding: original order
output_pe_orig, weights_pe_orig = scaled_dot_product_attention(
    embeddings_with_pe, embeddings_with_pe, embeddings_with_pe
)

# Self-attention WITH positional encoding: scrambled order
output_pe_scram, weights_pe_scram = scaled_dot_product_attention(
    scrambled_embeddings_with_pe, scrambled_embeddings_with_pe, scrambled_embeddings_with_pe
)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

w1 = weights_pe_orig.detach().numpy()
im1 = axes[0].imshow(w1, cmap='YlOrRd', aspect='equal', vmin=0)
axes[0].set_xticks(range(len(words)))
axes[0].set_xticklabels(words, rotation=45, ha='right')
axes[0].set_yticks(range(len(words)))
axes[0].set_yticklabels(words)
axes[0].set_title('With PE: Original Order')
for i in range(w1.shape[0]):
    for j in range(w1.shape[1]):
        color = 'white' if w1[i,j] > w1.max()*0.6 else 'black'
        axes[0].text(j, i, f'{w1[i,j]:.2f}', ha='center', va='center', fontsize=8, color=color)

w2 = weights_pe_scram.detach().numpy()
im2 = axes[1].imshow(w2, cmap='YlOrRd', aspect='equal', vmin=0)
axes[1].set_xticks(range(len(scrambled_words)))
axes[1].set_xticklabels(scrambled_words, rotation=45, ha='right')
axes[1].set_yticks(range(len(scrambled_words)))
axes[1].set_yticklabels(scrambled_words)
axes[1].set_title('With PE: Scrambled Order')
for i in range(w2.shape[0]):
    for j in range(w2.shape[1]):
        color = 'white' if w2[i,j] > w2.max()*0.6 else 'black'
        axes[1].text(j, i, f'{w2[i,j]:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.suptitle('Positional encoding breaks permutation equivariance!', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Quantify the difference
max_diff = 0
for i in range(len(words)):
    for j in range(len(words)):
        scram_i = (perm == i).nonzero().item()
        scram_j = (perm == j).nonzero().item()
        diff = abs(weights_pe_orig[i, j].item() - weights_pe_scram[scram_i, scram_j].item())
        max_diff = max(max_diff, diff)

print(f"\nMax attention weight difference (with PE): {max_diff:.4f}")
if max_diff > 0.01:
    print("The weights are now DIFFERENT! Positional encoding worked.")

### ✏️ Was your prediction correct?

1. Did positional encoding fix the permutation equivariance problem?

   *Yes — the attention weights are now measurably different between the original and scrambled orderings. The max difference is no longer zero.*

2. In your own words, *why* does adding PE fix the problem? What changed about the inputs to the dot product?

   *Without PE, the dot product between two words depends only on their learned embeddings, which are the same regardless of position. With PE, each embedding gets a position-dependent vector added to it before the dot product is computed. So "keys" at position 1 has a different input vector than "keys" at position 4. The dot products now depend on both word identity AND position, breaking permutation equivariance.*

---

## Part 3: Building a SelfAttention Module (15 min)

Now let's build self-attention as a proper PyTorch module with **learned** projection matrices $W^Q$, $W^K$, $W^V$.

This is the real thing — not simplified attention with Q = K = V = X, but attention where Q, K, V are computed via learned linear transformations.

Notice what the projections produce: Q is used once (for the current token's "question"), but **K and V are needed by every future token**. During generation, these get stored in the KV cache.

### Task 3.1: Implement `SelfAttention(nn.Module)`

Fill in the `__init__` and `forward` methods. Use the blueprint from the slides.

**Shape checkpoint:** If input $X$ has shape `(n, d_model)` and we project to dimension $d_k$:
- $Q = XW^Q$: shape `(n, d_k)`
- $K = XW^K$: shape `(n, d_k)`  
- $V = XW^V$: shape `(n, d_k)`
- Scores $= QK^\top / \sqrt{d_k}$: shape `(n, n)`
- Weights $= \text{softmax}(\text{scores})$: shape `(n, n)`
- Output $= \text{weights} \cdot V$: shape `(n, d_k)`

In [ ]:
class SelfAttention(nn.Module):
    """
    Single-head self-attention with learned Q, K, V projections.
    
    Args:
        d_model: input embedding dimension
        d_k: dimension of Q, K, V projections
    """
    def __init__(self, d_model, d_k):
        super().__init__()
        # Three nn.Linear layers for W^Q, W^K, W^V
        # Each projects from d_model dimensions to d_k dimensions
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_k, bias=False)
        self.scale = d_k ** 0.5
    
    def forward(self, X):
        """
        Args:
            X: input embeddings, shape (n, d_model)
        
        Returns:
            output: shape (n, d_k)
            weights: shape (n, n) — the attention weight matrix
        """
        # Project X into Q, K, V
        Q = self.W_Q(X)  # (n, d_k)
        K = self.W_K(X)  # (n, d_k)
        V = self.W_V(X)  # (n, d_k)
        
        # Compute scaled dot-product attention
        # Step 1: scores = Q @ K^T / sqrt(d_k)
        scores = Q @ K.T / self.scale  # (n, n)
        
        # Step 2: weights = softmax(scores, dim=-1)
        weights = F.softmax(scores, dim=-1)  # (n, n)
        
        # Step 3: output = weights @ V
        output = weights @ V  # (n, d_k)
        
        return output, weights

### Task 3.2: Test your module

Let's test it on the chicken sentence embeddings from Day 1.

In [ ]:
# Chicken sentence with real GloVe-50d embeddings
chicken_words = ['the', 'chicken', 'did', 'not', 'cross', 'the', 'road', 'because', 'it']

chicken_embeddings = get_embeddings(chicken_words, glove)

print(f'Input shape: {chicken_embeddings.shape}  ({len(chicken_words)} words, {chicken_embeddings.shape[1]} dims)')
print(f'\nNote: we split "didn\'t" into "did" and "not" since GloVe')
print(f'has separate entries for each. Same sentence, same meaning.')

In [ ]:
# Create the module and run it
d_model = 50   # GloVe-50d
d_k = 25       # projection dimension

torch.manual_seed(395)  # for reproducibility
sa = SelfAttention(d_model=d_model, d_k=d_k)

output, weights = sa(chicken_embeddings)

print(f'Input shape:   {chicken_embeddings.shape}')
print(f'Output shape:  {output.shape}')
print(f'Weights shape: {weights.shape}')
print(f'\nExpected output shape: ({len(chicken_words)}, {d_k})')
print(f'Expected weights shape: ({len(chicken_words)}, {len(chicken_words)})')

# Verify weights sum to 1
row_sums = weights.sum(dim=-1)
print(f'\nWeight row sums: {row_sums.detach().tolist()}')
print(f'(Should all be approx 1.0)')

### Task 3.3: Visualize the attention patterns

In [ ]:
plot_attention_weights(weights, title='Learned Self-Attention (random weights)',
                     row_labels=chicken_words, col_labels=chicken_words)

### ✏️ Observation

1. The weights are from a randomly initialized (untrained) model. Do the attention patterns look meaningful? Should they?

   *No, the patterns are essentially random — the weight matrices $W^Q$, $W^K$, $W^V$ have random values, so the Q/K dot products have no semantic meaning. This is expected. The model hasn't been trained, so there's no reason for the attention to reflect linguistic structure. Meaningful patterns (like "it" attending strongly to "chicken") would only emerge after training on data where such relationships matter.*

2. After training on a coreference task, what pattern would you *expect* to see in the row for "it"?

   *We'd expect the row for "it" to have high weight on "chicken" (the entity "it" refers to) and lower weight on function words like "The", "didn't", "the". The model would learn $W^Q$ and $W^K$ such that pronoun queries produce high dot products with antecedent keys.*

### Task 3.4: Self-attention WITH positional encoding

Now let's add positional encoding before feeding the embeddings into self-attention.

In [ ]:
# Generate PE for the chicken sentence
PE_chicken = sinusoidal_positional_encoding(max_len=len(chicken_words), d_model=d_model)

# Add PE to embeddings
chicken_with_pe = chicken_embeddings + PE_chicken

# Run through the SAME self-attention module
output_pe, weights_pe = sa(chicken_with_pe)

# Plot both side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, w, title in [(axes[0], weights.detach().numpy(), 'Without PE'),
                       (axes[1], weights_pe.detach().numpy(), 'With PE')]:
    im = ax.imshow(w, cmap='YlOrRd', aspect='equal', vmin=0)
    ax.set_xticks(range(len(chicken_words)))
    ax.set_xticklabels(chicken_words, rotation=45, ha='right')
    ax.set_yticks(range(len(chicken_words)))
    ax.set_yticklabels(chicken_words)
    ax.set_title(title)
    for i in range(w.shape[0]):
        for j in range(w.shape[1]):
            color = 'white' if w[i,j] > w.max()*0.6 else 'black'
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.suptitle('Effect of Positional Encoding on Attention Weights (GloVe-50d)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### ✏️ Reflection

1. How do the attention weights change when you add positional encoding? What pattern differences do you notice?

   *The weights shift — some pairs get more attention, others less. Even though the model is untrained, PE introduces a systematic bias based on position. Nearby tokens may get slightly more attention because their PE vectors are more similar (the PE for position 3 is closer to position 4 than to position 20, especially in the low-frequency dimensions).*

2. With PE, do nearby words tend to get more attention? Why might that happen?

   *Often yes. The PE vectors for adjacent positions are more similar than those for distant positions (the high-frequency components change, but the low-frequency components are nearly identical). Since we add PE to the embeddings before computing Q and K, nearby tokens end up with more similar Q/K vectors, leading to higher dot products and higher attention weights.*

3. **Connecting to the bigger picture:** In an RNN, position information is "free" because tokens are processed sequentially. In a transformer, we have to add it explicitly. What does the transformer get in exchange for this extra step?

   *Parallelism. An RNN must process tokens one at a time in order, which is inherently sequential. A transformer processes all tokens simultaneously — the entire attention matrix is computed in one parallel operation. This makes transformers much faster to train on GPUs. The cost is that position information doesn't come for free, so we have to add PE explicitly.*

---

## Part 4: The KV Cache Bottleneck (10 min)

You've now built self-attention from scratch. Let's zoom out and ask: **what does this cost at inference time?**

During **training**, all tokens are available at once — we compute Q, K, V for the whole sequence in one parallel operation. But during **generation** (producing text one token at a time), each new token needs to attend to all previous tokens. That means it needs the K and V vectors from every previous step.

**The KV cache** stores these K and V vectors so we don't recompute them. It's the dominant memory cost during inference, and it's the reason context windows have limits.

### Task 4.1: Implement a KV cache size calculator

Write a function that computes the total KV cache memory for a given model configuration.

In [ ]:
def kv_cache_size_gb(n_layers, n_heads, d_k, seq_len, bytes_per_param=2):
    """
    Calculate KV cache size in gigabytes.
    
    Args:
        n_layers: number of transformer layers (L)
        n_heads: number of attention heads per layer (h)
        d_k: dimension per head
        seq_len: sequence length (number of tokens cached)
        bytes_per_param: 2 for FP16, 4 for FP32, 1 for INT8
    
    Returns:
        size_gb: cache size in gigabytes
    
    The cache stores K and V for every layer, every head, every token:
        total_numbers = 2 (K and V) × n_layers × n_heads × d_k × seq_len
    """
    # Calculate total number of values stored in the cache
    total_numbers = 2 * n_layers * n_heads * d_k * seq_len
    
    # Convert to gigabytes (1 GB = 1024^3 bytes)
    size_gb = (total_numbers * bytes_per_param) / (1024 ** 3)
    
    return size_gb

### Task 4.2: Calculate cache sizes for real models

Fill in the table by running the calculator for each model.

**Before running:** predict which model will need the most cache. What's your intuition?

In [ ]:
# Model configurations (approximate)
models = {
    'Llama-2-7B': {
        'n_layers': 32,
        'n_heads': 32,
        'd_k': 128,       # d_model=4096, d_k = 4096/32 = 128
    },
    'GPT-3 (175B)': {
        'n_layers': 96,
        'n_heads': 96,
        'd_k': 128,       # d_model=12288, d_k = 12288/96 = 128
    },
    'Llama-3-8B': {
        'n_layers': 32,
        'n_heads': 32,
        'd_k': 128,       # d_model=4096, d_k = 4096/32 = 128
    },
}

context_lengths = [2048, 8192, 32768, 131072]

print(f"{'Model':<18s}", end='')
for ctx in context_lengths:
    label = f'{ctx//1024}K' if ctx >= 1024 else str(ctx)
    print(f'{label:>10s}', end='')
print()
print("-" * 58)

for name, cfg in models.items():
    print(f'{name:<18s}', end='')
    for ctx in context_lengths:
        size = kv_cache_size_gb(
            n_layers=cfg['n_layers'],
            n_heads=cfg['n_heads'],
            d_k=cfg['d_k'],
            seq_len=ctx,
            bytes_per_param=2  # FP16
        )
        print(f'{size:>9.2f}G', end='')
    print()

### ✏️ Analysis

1. An NVIDIA A100 GPU has 80 GB of memory. Roughly half is used by the model weights. For Llama-2-7B, what is the maximum context length you can fit on a single A100? Show your reasoning.

   *Budget: ~40 GB for KV cache. Llama-2-7B has L=32, h=32, d_k=128. Per-token cache = 2 × 32 × 32 × 128 × 2 bytes = 524,288 bytes ≈ 0.5 MB. At 0.5 MB/token, 40 GB / 0.5 MB = ~80,000 tokens. So roughly 80K context. (In practice it's somewhat less due to activations and other memory overhead, which is why Llama-2's context window was initially 4K — serving many concurrent users divides the memory budget.)*

2. GPT-3 has 3× more layers than Llama-2-7B. How does that affect the KV cache? Is it 3× larger? Why or why not?

   *Yes, it is 3× larger (per token). KV cache scales linearly with L. GPT-3 has L=96 vs Llama-2's L=32, so 3× more layers means 3× more K and V matrices to cache. Additionally, GPT-3 has d_model=12288 vs 4096, so it's also 3× wider — making the per-token cache about 9× larger overall.*

3. When a chatbot tells you "this conversation is too long, please start a new one" — what is actually happening at the hardware level?

   *The KV cache for the conversation has grown to consume the available GPU memory. Every token in the conversation (both user messages and model responses) has K and V vectors cached across all layers. When this total exceeds the memory budget, the system can't process additional tokens. Starting a new conversation clears the KV cache, freeing the GPU memory.*

### Task 4.3: Visualizing the cache growth

Let's plot how cache size grows with sequence length for different model sizes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

seq_lengths = list(range(0, 150_000, 1000))

for name, cfg in models.items():
    sizes = [kv_cache_size_gb(cfg['n_layers'], cfg['n_heads'], cfg['d_k'], n) 
             for n in seq_lengths]
    ax.plot([n/1000 for n in seq_lengths], sizes, label=name, linewidth=2)

# GPU memory reference lines
ax.axhline(y=40, color='red', linestyle='--', alpha=0.5, label='~Budget on 80GB GPU (50% for weights)')
ax.axhline(y=12, color='orange', linestyle='--', alpha=0.5, label='~Budget on 24GB GPU (50% for weights)')

ax.set_xlabel('Sequence Length (thousands of tokens)')
ax.set_ylabel('KV Cache Size (GB)')
ax.set_title('KV Cache Growth: Why Context Windows Have Limits')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Observations

1. The cache grows **linearly** with sequence length. But the attention *computation* grows **quadratically**. Which one is the binding constraint in practice, and why?

   *Memory (the KV cache) is typically the binding constraint, not compute. Flash Attention (Dao et al., 2022) reduced the memory footprint of the attention computation itself from O(n²) to O(n) through tiling and kernel fusion — but it doesn't reduce the KV cache, which still grows linearly with sequence length. Since the KV cache must be stored for the entire duration of generation (not just during one attention operation), it dominates GPU memory at long contexts.*

2. What strategies might you use to reduce the KV cache? (Brainstorm — we won't implement these, but think about what knobs you could turn.)

   *Several strategies exist: (1) Quantize the cache — store K/V in INT8 or INT4 instead of FP16, halving or quartering the memory. (2) Multi-Query Attention (MQA) — share K and V across all heads, so only one set of K/V is cached per layer instead of h sets. (3) Grouped-Query Attention (GQA) — a middle ground where groups of heads share K/V. (4) Sliding window attention — only cache the last N tokens instead of the full context. (5) Eviction policies — drop cache entries for tokens that consistently receive low attention.*

### Task 4.4: The per-token cost

Let's make this concrete: how much memory does *one additional token* cost?

In [ ]:
# Per-token KV cache cost
print("Per-token KV cache cost (FP16):")
print(f"{'Model':<18s} {'Cache/token':>15s} {'Equivalent':>20s}")
print("-" * 55)

for name, cfg in models.items():
    # 2 (K+V) × n_layers × n_heads × d_k × 2 bytes
    bytes_per_token = 2 * cfg['n_layers'] * cfg['n_heads'] * cfg['d_k'] * 2
    kb_per_token = bytes_per_token / 1024
    print(f"{name:<18s} {bytes_per_token:>12,d} B {kb_per_token:>16.0f} KB/token")

print()
print("Think about it: every token you type in a chatbot prompt,")
print("and every token the model generates in response,")
print("adds this much to the KV cache — for the entire conversation.")

---

## Part 5: Shape Trace — Multi-Head Attention (5 min)

We won't implement multi-head attention in this notebook (you'll do it on the homework), but let's make sure we understand the shapes and their cache implications.

### Task 5.1: Fill in the shapes

Given: $n = 10$ tokens, $d_{\text{model}} = 128$, $h = 8$ heads.

In [ ]:
# Multi-head attention shape trace
# Filled in with correct values

n = 10
d_model = 128
h = 8
d_k = d_model // h  # dimension per head

print("Multi-Head Attention Shape Trace")
print("=" * 55)
print(f"{'Step':<25s} {'Shape':>25s}")
print("-" * 55)
print(f"{'Input X':<25s} {'(' + str(n) + ', ' + str(d_model) + ')':>25s}")
print(f"{'Q_i per head':<25s} {'(' + str(n) + ', ' + str(d_k) + ')':>25s}")
print(f"{'Scores per head':<25s} {'(' + str(n) + ', ' + str(n) + ')':>25s}")
print(f"{'head_i output':<25s} {'(' + str(n) + ', ' + str(d_k) + ')':>25s}")
print(f"{'Concatenated':<25s} {'(' + str(n) + ', ' + str(d_model) + ')':>25s}")
print(f"{'After W^O':<25s} {'(' + str(n) + ', ' + str(d_model) + ')':>25s}")

print()
print(f"d_k = d_model / h = {d_model} / {h} = {d_k}")
print(f"\nKV cache per layer for these {n} tokens:")
print(f"  2 (K+V) x {h} heads x {d_k} dims x {n} tokens = {2*h*d_k*n} numbers")
print(f"  Note: h x d_k = {h*d_k} = d_model. More heads doesn't increase the cache!")

### ✏️ Checkpoint Question

If we change from $h = 8$ heads to $h = 4$ heads (keeping $d_{\text{model}} = 128$):

1. What is $d_k$ now?

   *$d_k = 128 / 4 = 32$. Each head now works with 32-dimensional Q, K, V vectors instead of 16.*

2. Does each head have more or less capacity? Why?

   *More capacity. Each head now operates in a 32-dimensional subspace instead of 16-dimensional. It can represent more complex patterns in its Q/K dot products. The tradeoff is that we have fewer heads (4 instead of 8), so the model learns fewer distinct attention patterns.*

3. Does the total KV cache size per layer change? Why or why not?

   *No, it stays exactly the same. KV cache per layer = $2 \times n \times h \times d_k = 2 \times n \times d_{\text{model}}$. Since $h \times d_k = d_{\text{model}} = 128$ regardless of how we split it (8×16 or 4×32), the total cache is unchanged. This is why the number of heads is "free" from a cache perspective — it's a capacity allocation decision, not a memory decision.*

---

## Exit Ticket — Model Answer

### Concept Map

Draw a concept map connecting: self-attention, multi-head attention, positional encoding, KV cache, query, key, value, context window, nn.Embedding, CNN filters, RNN sequential processing, permutation equivariance.

**Model answer (key connections):**

- **nn.Embedding** — *produces* → **static embeddings** — *which become input to* → **self-attention**
- **positional encoding** — *is added to embeddings to fix* → **permutation equivariance** — *which is the property that attention treats input as a* → **set, not a sequence**
- **self-attention** — *projects embeddings into* → **query, key, value** — *via learned $W^Q, W^K, W^V$*
- **query** — *is used once and discarded; but* → **key** and **value** — *are stored in the* → **KV cache** — *which determines the* → **context window** *limit*
- **multi-head attention** — *runs $h$ parallel attention heads, analogous to* → **CNN filters** — *which each learn different spatial patterns (Lab 5)*
- **RNN sequential processing** — *bakes in position for free but creates a bottleneck; transformers trade this for* → **parallelism** + explicit **positional encoding**
- **KV cache** — *scales as $2 \times L \times d_{\text{model}} \times n$; this is the binding constraint on* → **context window** *length*

There are many valid concept maps. The key test: can the student articulate *why* each connection exists, not just *that* it exists?